# Processed data check

Inspects one match from `data/processed/<match_id>/`, the Parquet tables written by
`scripts/extract_awpy.py`. No demo files are needed and nothing is re-parsed.

Run the cells in order.

## Imports

In [1]:
import os
from pathlib import Path

import pandas as pd

## Set current dir to project root dir

In [2]:
def find_project_root() -> Path:
    """Walk up from CWD until we find the project root."""
    markers = ["pyproject.toml", "uv.lock"]
    path = Path.cwd()

    while True:
        if all((path / marker).is_file() for marker in markers):
            return path
        if path == path.parent:
            raise FileNotFoundError("Could not find project root")
        path = path.parent


os.chdir(find_project_root())

## Choosing the match

Any folder name in `data/processed/` works. Note that
`falcons-vs-natus-vincere-m3-mirage` was recorded in two parts: its tables have a
`part` column (0 or 1), and tick numbers restart in each part.

In [3]:
# Select the match to analyze.
match_name = "mouz-vs-spirit-m1-dust2"
# match_name = "mouz-vs-spirit-m2-mirage"

match_dir = Path("data/processed") / match_name


def load(table: str) -> pd.DataFrame:
    """Read one Parquet table of the selected match."""
    return pd.read_parquet(match_dir / f"{table}.parquet")

## Tables available for the match

In [4]:
overview = pd.DataFrame(
    [
        {
            "table": path.stem,
            "rows": len(df := pd.read_parquet(path)),
            "columns": df.shape[1],
        }
        for path in sorted(match_dir.glob("*.parquet"))
    ]
)
overview

,table,rows,columns
0,blinds,219,21
1,bomb,128,13
2,chat,35,8
3,convars,64,3
4,damages,482,29
5,fires,93,14
6,grenades,58132,13
7,header,9,3
8,item_events,1581,17
9,kills,139,40


## Rounds and score

In [5]:
rounds = load("rounds")
players = load("players")

print("Rounds:", len(rounds))
print("Overtime rounds:", int(rounds["is_overtime"].sum()))
print("\nHow rounds ended:")
print(rounds["reason_name"].value_counts(dropna=False))

rounds[["round_num", "winner_side", "reason_name", "freeze_end_tick", "end_tick"]]

Rounds: 20
Overtime rounds: 0

How rounds ended:
reason_name
ct_win            6
terrorists_win    5
target_bombed     4
bomb_defused      4
target_saved      1
Name: count, dtype: int64


,round_num,winner_side,reason_name,freeze_end_tick,end_tick
0,1,terrorist,terrorists_win,1272,6277
1,2,terrorist,target_bombed,7877,16279
2,3,counter-terrorist,bomb_defused,17879,27266
3,4,counter-terrorist,ct_win,28866,33401
4,5,counter-terrorist,ct_win,35001,38370
5,6,counter-terrorist,target_saved,42078,49438
6,7,counter-terrorist,bomb_defused,51038,59229
7,8,counter-terrorist,ct_win,63324,69827
8,9,counter-terrorist,ct_win,71427,77827
9,10,counter-terrorist,bomb_defused,79427,87538


## Snapshots overview

One row per player every second of live play, dead players included (with `health` 0).

In [6]:
snapshots = load("snapshots")

print("Shape:", snapshots.shape)
print("Rounds:", snapshots["round_num"].nunique())
print("Players:", snapshots["steamid"].nunique())
print("First tick:", snapshots["tick"].min())
print("Last tick:", snapshots["tick"].max())
print("Teams:", sorted(snapshots["team_clan_name"].dropna().unique()))
print(sorted(snapshots["name"].dropna().unique()))

Shape: (20720, 43)
Rounds: 20
Players: 10
First tick: 1272
Last tick: 189303
Teams: ['MOUZ', 'Team Spirit']
['PR', 'Spinx', 'donk', 'magixx', 'sh1ro', 'tN1R', 'torzsi', 'xelex', 'xertioN', 'zont1x']


In [7]:
print("Rows per round:")
print(snapshots.groupby("round_num").size())

print("\nMissing values (columns with any):")
missing = snapshots.isna().sum()
print(missing[missing > 0])

print("\nHealth summary:")
print(snapshots["health"].describe())

Rows per round:
round_num
1      790
2     1320
3     1470
4      710
5      530
6     1150
7     1280
8     1020
9     1000
10    1270
11     920
12    1140
13     860
14     620
15    1210
16     820
17    1390
18    1170
19     750
20    1300
dtype: int64

Missing values (columns with any):
steamid             4914
name                4914
active_weapon       5174
primary_weapon      7985
secondary_weapon    5174
team_clan_name      4914
dtype: int64

Health summary:
count    20720.000000
mean        70.371766
std         43.390882
min          0.000000
25%          2.000000
50%        100.000000
75%        100.000000
max        100.000000
Name: health, dtype: float64


In [8]:
snapshots.columns.tolist()

['match_id',
 'tick',
 'steamid',
 'name',
 'side',
 'x',
 'y',
 'z',
 'velocity_x',
 'velocity_y',
 'velocity_z',
 'velocity',
 'pitch',
 'yaw',
 'health',
 'armor',
 'has_helmet',
 'has_defuser',
 'has_bomb',
 'active_weapon',
 'primary_weapon',
 'secondary_weapon',
 'fire_grenades',
 'smoke_grenades',
 'he_grenades',
 'flashbangs',
 'decoy_grenades',
 'equipment_value',
 'equipment_value_round_start',
 'money',
 'is_crouched',
 'is_walking',
 'is_jumping',
 'is_in_bomb_zone',
 'is_scoped',
 'is_reloading',
 'is_silent_reloading',
 'is_defusing',
 'is_blinded',
 'flash_duration',
 'inventory',
 'round_num',
 'team_clan_name']

## First snapshots of round 1

In [9]:
round_1 = snapshots[snapshots["round_num"] == 1]

round_1[
    [
        "tick",
        "name",
        "side",
        "health",
        "armor",
        "money",
        "equipment_value",
        "primary_weapon",
        "secondary_weapon",
        "x",
        "y",
        "z",
    ]
].head(20)

,tick,name,side,health,armor,money,equipment_value,primary_weapon,secondary_weapon,x,y,z
0,1272,xelex,counter-terrorist,100,100,150,1150,NaN,usp_silencer,160.122742,2369.676270,-119.918701
1,1272,torzsi,counter-terrorist,100,100,150,850,NaN,usp_silencer,182.249908,2439.011719,-120.968750
2,1272,tN1R,terrorist,100,100,150,1250,NaN,glock,-493.000000,-808.000000,108.563370
3,1272,magixx,terrorist,100,0,100,1100,NaN,glock,-696.844604,-806.623718,116.708206
4,1272,donk,terrorist,100,100,150,850,NaN,glock,-1141.000000,-808.000000,116.685989
5,1272,zont1x,terrorist,100,100,150,1450,NaN,glock,-857.506531,-738.361328,122.089119
6,1272,PR,counter-terrorist,100,100,150,850,NaN,usp_silencer,351.392120,2352.942383,-120.474915
7,1272,sh1ro,terrorist,100,0,100,900,NaN,glock,-367.000000,-808.000000,83.744965
8,1272,Spinx,counter-terrorist,100,0,0,700,NaN,usp_silencer,334.368744,2433.733643,-120.362549
9,1272,xertioN,counter-terrorist,100,100,150,950,NaN,elite,258.159393,2480.553711,-121.076660


## Team state per snapshot

Dead players are in the snapshots with `health` 0, so only players with health above 0
count as alive. A side with nobody alive has no rows left after that filter; the pivot
fills it with 0.

In [10]:
alive_players = snapshots[snapshots["health"] > 0]

team_stats = (
    alive_players.groupby(["round_num", "tick", "side"])
    .agg(
        alive_count=("steamid", "nunique"),
        total_health=("health", "sum"),
        avg_health=("health", "mean"),
        equipment_value=("equipment_value", "sum"),
    )
    .reset_index()
)

team_features = team_stats.pivot(
    index=["round_num", "tick"],
    columns="side",
    values=["alive_count", "total_health", "avg_health", "equipment_value"],
)
side_short = {"terrorist": "t", "counter-terrorist": "ct"}
team_features.columns = [
    f"{metric}_{side_short[side]}" for metric, side in team_features.columns
]
team_features = team_features.reset_index()
count_cols = [
    c
    for c in team_features.columns
    if c.startswith(("alive_count", "total_health", "equipment_value"))
]
team_features[count_cols] = team_features[count_cols].fillna(0)

team_features.head()

,round_num,tick,alive_count_ct,alive_count_t,total_health_ct,total_health_t,avg_health_ct,avg_health_t,equipment_value_ct,equipment_value_t
0,1,1272,5.0,5.0,500.0,500.0,100.0,100.0,4500.0,5550.0
1,1,1336,5.0,5.0,500.0,500.0,100.0,100.0,4500.0,5550.0
2,1,1400,5.0,5.0,500.0,500.0,100.0,100.0,4400.0,5550.0
3,1,1464,5.0,5.0,500.0,500.0,100.0,100.0,4400.0,5550.0
4,1,1528,5.0,5.0,500.0,500.0,100.0,100.0,4400.0,5550.0


## Snapshots where players have died or taken damage

In [11]:
cols = [
    "round_num",
    "tick",
    "alive_count_t",
    "alive_count_ct",
    "total_health_t",
    "total_health_ct",
]

team_features[
    (team_features["alive_count_t"] < 5)
    | (team_features["alive_count_ct"] < 5)
    | (team_features["total_health_t"] < 500)
    | (team_features["total_health_ct"] < 500)
][cols].head(20)

,round_num,tick,alive_count_t,alive_count_ct,total_health_t,total_health_ct
20,1,2552,5.0,5.0,500.0,403.0
21,1,2616,5.0,5.0,500.0,403.0
22,1,2680,5.0,5.0,500.0,403.0
23,1,2744,5.0,5.0,500.0,403.0
24,1,2808,5.0,5.0,486.0,403.0
25,1,2872,4.0,5.0,400.0,403.0
26,1,2936,4.0,5.0,400.0,403.0
27,1,3000,4.0,5.0,400.0,403.0
28,1,3064,4.0,5.0,400.0,403.0
29,1,3128,4.0,5.0,400.0,403.0


## Kills

In [12]:
kills = load("kills")
live_kills = kills[kills["in_play"]]

print("Kills during live play:", len(live_kills))
print("\nKills per round:")
print(live_kills.groupby("round_num").size())

live_kills[
    [
        "round_num",
        "tick",
        "attacker_name",
        "attacker_side",
        "victim_name",
        "victim_side",
        "weapon",
    ]
].head(10)

Kills during live play: 138

Kills per round:
round_num
1     9
2     9
3     9
4     5
5     5
6     4
7     9
8     5
9     6
10    7
11    7
12    8
13    7
14    5
15    8
16    8
17    7
18    4
19    7
20    9
dtype: int64


,round_num,tick,attacker_name,attacker_side,victim_name,victim_side,weapon
0,1,2821,torzsi,counter-terrorist,tN1R,terrorist,usp_silencer
1,1,3233,sh1ro,terrorist,torzsi,counter-terrorist,glock
2,1,3550,Spinx,counter-terrorist,sh1ro,terrorist,usp_silencer
3,1,3621,magixx,terrorist,xertioN,counter-terrorist,glock
4,1,5549,Spinx,counter-terrorist,magixx,terrorist,usp_silencer
5,1,5560,zont1x,terrorist,xelex,counter-terrorist,glock
6,1,5736,PR,counter-terrorist,zont1x,terrorist,usp_silencer
7,1,5792,donk,terrorist,PR,counter-terrorist,glock
8,1,6277,donk,terrorist,Spinx,counter-terrorist,glock
9,2,11486,donk,terrorist,xertioN,counter-terrorist,ak47
